# 🫁 Phase 1.1v2: Enhanced Image Classifier (PA Viewpoint)

### Upgrades over Phase 1.1:
1. **Medical Pre-trained Backbone**: DenseNet121 pretrained on 8 CXR datasets (CheXpert, MIMIC-CXR, NIH, PadChest, etc.) via `torchxrayvision`
2. **CLAHE Preprocessing**: Contrast Limited Adaptive Histogram Equalization to normalize variable X-ray contrast
3. **Lung ROI Cropping**: Auto-detected lung bounding box to focus network on lung tissue
4. **Safe Augmentations Only**: No horizontal flip (which would flip heart anatomy); only medically valid transforms
5. **CBAM Attention**: Channel-Spatial attention on top of CXR-pretrained backbone

### Pipeline:
```
CXR Image → CLAHE → Lung ROI Crop → DenseNet121 (CXR-pretrained) → CBAM → Classifier
```

In [ ]:
# --- CELL 0: Imports & Reproducibility ---
import os, random, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score, f1_score, auc
)

import torchxrayvision as xrv

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('Memory:', torch.cuda.get_device_properties(DEVICE).total_memory / 1e9, 'GB')

In [ ]:
# --- CELL 1: Paths & Hyperparameters ---
BASE_DIR   = r'C:\2026\PneumoFusionNet\mimic\1000_dataset'
CSV_PATH   = os.path.join(BASE_DIR, 'mimic_paired_dataset.csv')
SAVE_DIR   = os.path.join(BASE_DIR, 'outputs', 'Phase_1.1v2_PA_enhanced')
MODEL_PATH = os.path.join(SAVE_DIR, 'best_enhanced_model.pth')
BBOX_CSV   = os.path.join(SAVE_DIR, 'lung_bboxes.csv')  # Pre-computed lung bounding boxes

IMG_SIZE      = 224
BATCH_SIZE    = 8
NUM_EPOCHS    = 40
LR            = 1e-4
WEIGHT_DECAY  = 1e-4
NUM_CLASSES   = 2
WARMUP_EPOCHS = 3
GRAD_CLIP     = 1.0
PATIENCE      = 10

CLASSES = ['NORMAL', 'PNEUMONIA']

os.makedirs(SAVE_DIR, exist_ok=True)
print('Save dir :', SAVE_DIR)

In [ ]:
# --- CELL 2: Data Loading & Filtering (PA, Balanced) ---
df = pd.read_csv(CSV_PATH)

def resolve_data_path(path):
    if pd.isna(path) or str(path).strip() == '':
        return ''
    path = str(path).replace('/', os.sep).replace('\\', os.sep)
    return path if os.path.isabs(path) else os.path.join(BASE_DIR, path)

df['image_path'] = df['image_path'].apply(resolve_data_path)

df['label'] = np.select(
    [df['pneumonia_label'].eq(1.0), df['no_finding_label'].eq(1.0)],
    [1, 0],
    default=-1
)

df = df[(df['label'] >= 0) & df['view_position'].eq('PA')].copy()
df['label_name'] = df['label'].map({0: 'Normal', 1: 'Pneumonia'})

# Balanced sampling
class_counts = df['label'].value_counts()
n_per_class  = int(class_counts.min())
print(f'Balancing: {n_per_class} per class')

normal_df    = df[df['label'] == 0].sample(n=n_per_class, random_state=SEED)
pneumonia_df = df[df['label'] == 1].sample(n=n_per_class, random_state=SEED)
df = pd.concat([normal_df, pneumonia_df]).sample(frac=1, random_state=SEED).reset_index(drop=True)

missing = df['image_path'].apply(lambda p: not os.path.exists(p)).sum()
print(f'Total: {len(df)} | Missing images: {missing}')

In [ ]:
# --- CELL 3: Pre-compute Lung Bounding Boxes ---
# Uses torchxrayvision's built-in processing to find lung regions.
# This cell only needs to run ONCE — results are cached to CSV.

from tqdm import tqdm

def compute_lung_bbox(image_path, margin=10):
    """
    Compute lung bounding box using intensity-based thresholding.
    For CXRs, the lung regions are darker (lower pixel values) than the
    surrounding anatomy. We use Otsu's thresholding to find the lung region.
    
    Returns: (x_min, y_min, x_max, y_max) or None if detection fails.
    """
    try:
        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            return None
        
        h, w = img.shape
        
        # Apply CLAHE for better contrast before thresholding
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        img_eq = clahe.apply(img)
        
        # Gaussian blur to reduce noise
        blurred = cv2.GaussianBlur(img_eq, (5, 5), 0)
        
        # Otsu's thresholding to separate lung from background
        _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
        # Find contours
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        if not contours:
            return None
        
        # Filter contours by area (lung regions are large)
        min_area = h * w * 0.05  # At least 5% of image area
        lung_contours = [c for c in contours if cv2.contourArea(c) > min_area]
        
        if not lung_contours:
            # Fallback: use center crop (80% of image)
            cx, cy = w // 2, h // 2
            crop_w, crop_h = int(w * 0.8), int(h * 0.8)
            return (cx - crop_w // 2, cy - crop_h // 2, cx + crop_w // 2, cy + crop_h // 2)
        
        # Combine all lung contours into one bounding box
        all_points = np.vstack(lung_contours)
        x, y, bw, bh = cv2.boundingRect(all_points)
        
        # Add margin
        x_min = max(0, x - margin)
        y_min = max(0, y - margin)
        x_max = min(w, x + bw + margin)
        y_max = min(h, y + bh + margin)
        
        return (x_min, y_min, x_max, y_max)
        
    except Exception as e:
        return None

if os.path.exists(BBOX_CSV):
    print(f'Loading pre-computed bounding boxes from {BBOX_CSV}')
    bbox_df = pd.read_csv(BBOX_CSV)
else:
    print('Computing lung bounding boxes for all images...')
    records = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        bbox = compute_lung_bbox(row['image_path'])
        if bbox is not None:
            records.append({
                'image_path': row['image_path'],
                'x_min': bbox[0], 'y_min': bbox[1],
                'x_max': bbox[2], 'y_max': bbox[3]
            })
        else:
            # Fallback: full image
            records.append({
                'image_path': row['image_path'],
                'x_min': 0, 'y_min': 0,
                'x_max': -1, 'y_max': -1  # sentinel for 'use full image'
            })
    bbox_df = pd.DataFrame(records)
    bbox_df.to_csv(BBOX_CSV, index=False)
    print(f'Saved {len(bbox_df)} bounding boxes to {BBOX_CSV}')

# Merge bboxes into df
bbox_lookup = bbox_df.set_index('image_path').to_dict('index')
print(f'Bounding boxes loaded for {len(bbox_lookup)} images.')

In [ ]:
# --- CELL 4: Visualize CLAHE + Lung ROI Effect ---
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('CLAHE + Lung ROI Preprocessing', fontsize=14, fontweight='bold')

sample_rows = df.sample(4, random_state=SEED)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

for col, (_, row) in enumerate(sample_rows.iterrows()):
    img_raw = cv2.imread(row['image_path'], cv2.IMREAD_GRAYSCALE)
    img_clahe = clahe.apply(img_raw)
    
    # Get bbox
    bb = bbox_lookup.get(row['image_path'], None)
    
    # Top row: Raw image
    axes[0][col].imshow(img_raw, cmap='gray')
    axes[0][col].set_title(f'Raw - {row["label_name"]}', fontsize=9)
    axes[0][col].axis('off')
    if bb and bb['x_max'] > 0:
        import matplotlib.patches as patches
        rect = patches.Rectangle((bb['x_min'], bb['y_min']),
                                  bb['x_max']-bb['x_min'], bb['y_max']-bb['y_min'],
                                  linewidth=2, edgecolor='lime', facecolor='none')
        axes[0][col].add_patch(rect)
    
    # Bottom row: CLAHE + Cropped
    if bb and bb['x_max'] > 0:
        cropped = img_clahe[bb['y_min']:bb['y_max'], bb['x_min']:bb['x_max']]
    else:
        cropped = img_clahe
    axes[1][col].imshow(cropped, cmap='gray')
    axes[1][col].set_title(f'CLAHE + Lung ROI', fontsize=9)
    axes[1][col].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'preprocessing_comparison.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- CELL 5: Patient-Level Split ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=14)
train_idx, temp_idx = next(gss.split(df, groups=df['subject_id']))
train_df = df.iloc[train_idx].reset_index(drop=True)
temp_df = df.iloc[temp_idx].reset_index(drop=True)

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=4)
val_idx, test_idx = next(gss_val.split(temp_df, groups=temp_df['subject_id']))
val_df   = temp_df.iloc[val_idx].reset_index(drop=True)
test_df  = temp_df.iloc[test_idx].reset_index(drop=True)

print('=' * 60)
print('DATASET SPLIT SUMMARY (Patient-Level)')
print('=' * 60)
for split_name, split_df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    n_total    = len(split_df)
    n_pneumonia = split_df['label'].sum()
    n_normal   = (split_df['label'] == 0).sum()
    print(f'{split_name:6s} | Total: {n_total:3d} | Normal: {n_normal:3d} | Pneumonia: {n_pneumonia:3d}')
print('=' * 60)

In [ ]:
# --- CELL 6: Normalization Stats (from CLAHE+Cropped training images) ---
print('Computing normalization stats from CLAHE-processed training set...')
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
means = []
stds  = []

for idx in range(min(len(train_df), 200)):
    try:
        img = cv2.imread(train_df.iloc[idx]['image_path'], cv2.IMREAD_GRAYSCALE)
        img = clahe.apply(img)
        
        # Apply lung crop if available
        bb = bbox_lookup.get(train_df.iloc[idx]['image_path'], None)
        if bb and bb['x_max'] > 0:
            img = img[bb['y_min']:bb['y_max'], bb['x_min']:bb['x_max']]
        
        img_array = np.array(img, dtype=np.float32) / 255.0
        means.append(img_array.mean())
        stds.append(img_array.std())
    except:
        continue

MEAN = [np.mean(means)]
STD  = [np.mean(stds)]
print(f'CLAHE-adjusted MEAN: {MEAN[0]:.4f}')
print(f'CLAHE-adjusted STD : {STD[0]:.4f}')

In [ ]:
# --- CELL 7: Safe Augmentation Transforms ---
# NO horizontal flip (flips heart anatomy)
# Only medically valid transforms

tfms = {
    'train': transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.9, 1.0), ratio=(0.95, 1.05)),
        transforms.RandomRotation(7),          # Subtle rotation only
        transforms.ColorJitter(brightness=0.15, contrast=0.15),  # X-ray exposure variation
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5)),  # Slight defocus
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD)
    ]),
    'val': transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD)
    ]),
    'test': transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD)
    ]),
}
print('Safe augmentation transforms ready (NO horizontal flip).')

In [ ]:
# --- CELL 8: Enhanced Dataset with CLAHE + Lung ROI ---
class EnhancedCXRDataset(Dataset):
    """
    Dataset with CLAHE preprocessing and Lung ROI cropping.
    """
    def __init__(self, df, bbox_lookup, transform=None):
        self.df        = df.reset_index(drop=True)
        self.bbox_lookup = bbox_lookup
        self.transform = transform
        self.clahe     = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # 1. Load grayscale image
        img = cv2.imread(row.image_path, cv2.IMREAD_GRAYSCALE)
        
        # 2. Apply CLAHE contrast normalization
        img = self.clahe.apply(img)
        
        # 3. Lung ROI crop
        bb = self.bbox_lookup.get(row.image_path, None)
        if bb and bb['x_max'] > 0:
            img = img[bb['y_min']:bb['y_max'], bb['x_min']:bb['x_max']]
        
        # 4. Convert to PIL (for torchvision transforms)
        img = Image.fromarray(img)
        
        if self.transform:
            img = self.transform(img)
        
        label = int(row.label)
        return img, label


# Build dataloaders
train_labels  = train_df['label'].values
class_counts  = np.bincount(train_labels)
sample_weights = [1.0 / class_counts[l] for l in train_labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

dataloaders = {
    'train': DataLoader(
        EnhancedCXRDataset(train_df, bbox_lookup, tfms['train']),
        batch_size=BATCH_SIZE, sampler=sampler, num_workers=0
    ),
    'val': DataLoader(
        EnhancedCXRDataset(val_df, bbox_lookup, tfms['val']),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0
    ),
    'test': DataLoader(
        EnhancedCXRDataset(test_df, bbox_lookup, tfms['test']),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0
    ),
}

# Verify batch shape
sample_batch, sample_labels = next(iter(dataloaders['train']))
print(f'Batch shape: {sample_batch.shape} | Labels: {sample_labels}')

In [ ]:
# --- CELL 9: CBAM Attention Module ---
class ChannelAttention(nn.Module):
    """Channel attention sub-module of CBAM."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        avg_out = self.mlp(self.avg_pool(x))
        max_out = self.mlp(self.max_pool(x))
        return x * self.sigmoid(avg_out + max_out)

class SpatialAttention(nn.Module):
    """Spatial attention sub-module of CBAM."""
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        avg_out = x.mean(dim=1, keepdim=True)
        max_out = x.max(dim=1, keepdim=True)[0]
        spatial = torch.cat([avg_out, max_out], dim=1)
        return x * self.sigmoid(self.conv(spatial))

class CBAM(nn.Module):
    """Convolutional Block Attention Module (CBAM)."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.channel_att = ChannelAttention(channels, reduction)
        self.spatial_att = SpatialAttention()
    
    def forward(self, x):
        x = self.channel_att(x)
        x = self.spatial_att(x)
        return x

print('CBAM module defined.')

In [ ]:
# --- CELL 10: Enhanced Model Architecture ---
class EnhancedPneumoNet(nn.Module):
    """
    DenseNet121 (CXR-pretrained via torchxrayvision)
    + CBAM Attention
    + Custom Classifier Head
    
    Feature dim: 1024 (same as original PneumoFusionNet for Phase 2 compatibility)
    """
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        
        # Load CXR-pretrained DenseNet121 from torchxrayvision
        print('Loading DenseNet121 pretrained on 8 CXR datasets...')
        xrv_model = xrv.models.DenseNet(weights='densenet121-res224-all')
        
        # Extract the feature layers (everything except final classifier)
        self.features = xrv_model.features
        
        # Freeze early layers (denseblock1, denseblock2, denseblock3)
        # Fine-tune denseblock4 + norm5
        freeze_until = 'denseblock3'  # Freeze up to and including denseblock3
        frozen = True
        for name, child in self.features.named_children():
            if frozen:
                for p in child.parameters():
                    p.requires_grad = False
            if name == freeze_until:
                frozen = False  # Start unfreezing after this block
        
        # CBAM attention on the 1024-d feature maps
        self.cbam = CBAM(1024, reduction=16)
        
        # Pooling
        self.pool = nn.AdaptiveAvgPool2d(1)
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        # DenseNet expects 1-channel grayscale (torchxrayvision handles this)
        feats = self.features(x)          # [B, 1024, 7, 7]
        feats = F.relu(feats, inplace=True)
        feats = self.cbam(feats)           # [B, 1024, 7, 7] — attention-refined
        feats = self.pool(feats).flatten(1)  # [B, 1024]
        return self.classifier(feats)      # [B, 2]
    
    def get_features(self, x):
        """Extract 1024-d embeddings for Phase 2 multimodal fusion."""
        feats = self.features(x)
        feats = F.relu(feats, inplace=True)
        feats = self.cbam(feats)
        return self.pool(feats).flatten(1)  # [B, 1024]


model = EnhancedPneumoNet().to(DEVICE)

# Model stats
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total:,}')
print(f'Trainable params: {trainable:,} ({trainable/total*100:.1f}%)')

# Sanity check
dummy = torch.randn(2, 1, 224, 224).to(DEVICE)
print(f'Output shape  : {model(dummy).shape}')          # [2, 2]
print(f'Feature shape : {model.get_features(dummy).shape}')  # [2, 1024]

In [ ]:
# --- CELL 11: Loss, Optimizer & Scheduler ---
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                        lr=LR, weight_decay=WEIGHT_DECAY)

linear_warmup = LinearLR(optimizer, start_factor=0.1, total_iters=WARMUP_EPOCHS)
cosine_anneal = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - WARMUP_EPOCHS)
scheduler     = SequentialLR(
    optimizer,
    schedulers=[linear_warmup, cosine_anneal],
    milestones=[WARMUP_EPOCHS]
)
print('Optimizer, Loss, Scheduler configured.')

In [ ]:
# --- CELL 12: Training Loop ---
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for imgs, labels in dataloader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs      = model(imgs)
            loss         = criterion(outputs, labels)
            total_loss  += loss.item() * len(labels)
            probs        = F.softmax(outputs, dim=1)[:, 1]
            all_preds.extend(outputs.argmax(1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
            all_probs.extend(probs.cpu().tolist())

    avg_loss  = total_loss / len(dataloader.dataset)
    accuracy  = accuracy_score(all_labels, all_preds)
    auc_score = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.0
    return avg_loss, accuracy, auc_score, all_preds, all_labels, all_probs

history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_auc': []}
best_val_auc     = 0.0
best_epoch       = 0
patience_counter = 0

print('\n' + '='*80)
print('TRAINING STARTED — Enhanced PneumoNet (CXR-Pretrained DenseNet121 + CBAM)')
print('='*80)

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = train_correct = train_n = 0

    for imgs, labels in dataloaders['train']:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss    = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()
        train_loss    += loss.item() * len(labels)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_n       += len(labels)

    scheduler.step()
    train_loss_avg = train_loss / train_n
    train_acc      = train_correct / train_n

    val_loss, val_acc, val_auc, _, _, _ = evaluate(
        model, dataloaders['val'], criterion, DEVICE
    )

    history['train_loss'].append(train_loss_avg)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)

    if val_auc > best_val_auc:
        best_val_auc     = val_auc
        best_epoch       = epoch
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_PATH)
        marker = ' <- BEST'
    else:
        patience_counter += 1
        marker = ''

    if (epoch + 1) % 5 == 0 or epoch == 0 or marker != '':
        print(f'Epoch [{epoch+1:2d}/{NUM_EPOCHS}] | '
              f'Train Loss: {train_loss_avg:.4f} ({train_acc:.3f}) | '
              f'Val Loss: {val_loss:.4f} ({val_acc:.3f}) | '
              f'Val AUC: {val_auc:.4f}{marker}')

    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs)')
        break

print('='*80)
print(f'Training complete. Best val AUC: {best_val_auc:.4f} at epoch {best_epoch+1}')
print(f'Model saved to: {MODEL_PATH}')
print('='*80)

In [ ]:
# --- CELL 13: Training History Plot ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Enhanced PneumoNet — Training History', fontsize=14, fontweight='bold')

axes[0].plot(history['train_loss'], label='Train', lw=2, alpha=0.7)
axes[0].plot(history['val_loss'],   label='Val',   lw=2, alpha=0.7)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history['val_acc'], lw=2, color='green', label='Val Acc')
axes[1].axvline(best_epoch, color='red', ls='--', alpha=0.5, label=f'Best: {best_epoch+1}')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Val Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(history['val_auc'], lw=2, color='orange', label='Val AUC')
axes[2].axvline(best_epoch, color='red', ls='--', alpha=0.5, label=f'Best: {best_epoch+1}')
axes[2].axhline(best_val_auc, color='red', ls=':', alpha=0.5)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('AUC')
axes[2].set_title('Val AUC'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_history.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- CELL 14: Test Set Evaluation ---
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

test_loss, test_acc, test_auc, test_preds, test_labels, test_probs = evaluate(
    model, dataloaders['test'], criterion, DEVICE
)
test_f1 = f1_score(test_labels, test_preds, average='macro')

print('\n' + '='*60)
print('TEST SET RESULTS — Enhanced PneumoNet v2')
print('='*60)
print(f'Loss      : {test_loss:.4f}')
print(f'Accuracy  : {test_acc:.4f}')
print(f'AUC       : {test_auc:.4f}')
print(f'F1 (macro): {test_f1:.4f}')
print('\nClassification Report:')
print(classification_report(test_labels, test_preds, target_names=CLASSES))
print('='*60)

In [ ]:
# --- CELL 15: Confusion Matrix ---
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES,
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.title('Confusion Matrix — Enhanced PneumoNet v2', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'confusion_matrix.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- CELL 16: ROC Curve ---
fpr, tpr, _ = roc_curve(test_labels, test_probs)
roc_auc     = auc(fpr, tpr)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2.5, label=f'Enhanced v2 AUC = {roc_auc:.4f}')
plt.plot([0, 1], [0, 1], 'navy', lw=1.5, ls='--', label='Random')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve — Enhanced PneumoNet v2', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'roc_curve.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- CELL 17: Feature Extraction for Phase 2 ---
print('\nExtracting 1024-d image features split-wise...')
model.eval()

for split_name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    split_loader = DataLoader(
        EnhancedCXRDataset(split_df, bbox_lookup, tfms['val']),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0
    )

    feats, labs = [], []
    with torch.no_grad():
        for imgs, labels in split_loader:
            feat = model.get_features(imgs.to(DEVICE)).cpu()
            feats.append(feat)
            labs.extend(labels.tolist())

    img_features = torch.cat(feats)
    feat_path = os.path.join(SAVE_DIR, f'image_features_{split_name}.pt')
    torch.save({'features': img_features, 'labels': torch.tensor(labs)}, feat_path)
    print(f'{split_name:6s} | Features shape: {img_features.shape} | Saved to {feat_path}')

    meta_df          = split_df[['subject_id', 'study_id', 'label', 'label_name']].copy()
    meta_df['split'] = split_name
    meta_path        = feat_path.replace('.pt', '_meta.csv')
    meta_df.to_csv(meta_path, index=False)

print('\nPhase 1.1v2 COMPLETE. Features ready for Phase 2 multimodal fusion.')
print(f'All outputs saved to: {SAVE_DIR}')